# Synthetic Time Series Dataset Generator & Forecasting Demo

This notebook demonstrates the synthetic time series dataset and compares forecasting models (Moving Average vs. Naive Last-Value baseline) across diverse configurations including varying sequence lengths, noise levels, and trend types.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'seaborn==0.13.2')

In [ ]:
import json
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# NumPy 2.0 compatibility shims if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-0ece95-adaptive-smoothing-and-persistence-trade/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data_payload = load_data()
print("Loaded dataset successfully!")

In [ ]:
# Configuration parameters
WINDOW_SIZE = 3
MAX_EXAMPLES = 10

## Process and Evaluate Time Series Forecasting

In [ ]:
datasets = data_payload.get("datasets", [])
examples = []
if datasets:
    examples = datasets[0].get("examples", [])

results = []
for idx, ex in enumerate(examples[:MAX_EXAMPLES]):
    series = json.loads(ex["input"])
    base = json.loads(ex["output"])
    noise = ex["metadata_noise_level"]
    trend = ex["metadata_trend_type"]
    
    # Simple Moving Average forecast (window=WINDOW_SIZE)
    ma_forecast = []
    for i in range(len(series)):
        if i < WINDOW_SIZE:
            ma_forecast.append(series[i])
        else:
            ma_forecast.append(np.mean(series[i-WINDOW_SIZE:i]))
            
    # Naive last-value baseline forecast
    naive_forecast = [series[0]] + series[:-1]
    
    # Compute MSE
    series_arr = np.array(series)
    ma_mse = np.mean((series_arr - np.array(ma_forecast)) ** 2)
    naive_mse = np.mean((series_arr - np.array(naive_forecast)) ** 2)
    
    results.append({
        "id": ex["metadata_id"],
        "trend": trend,
        "noise": noise,
        "ma_mse": ma_mse,
        "naive_mse": naive_mse
    })

df_results = pd.DataFrame(results)
print(df_results.head())

## Visualization of Results

In [ ]:
plt.figure(figsize=(10, 5))
if len(examples) > 0:
    ex = examples[0]
    series = json.loads(ex["input"])
    plt.plot(series, label="Original Series", marker="o")
    plt.title(f"Synthetic Time Series Sample (Trend: {ex['metadata_trend_type']}, Noise: {ex['metadata_noise_level']})")
    plt.xlabel("Time Step")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.show()